# Session 1 — Solutions

**Try the exercises first.** Reading a solution produces the feeling of understanding without the
thing itself; the feeling wears off by Saturday and the gap doesn't.

Every solution below explains *why*, not just *what*, and each ends with where the idea comes back
later in the course.

In [1]:
import base64
import os
from pathlib import Path

import yaml
from dotenv import load_dotenv
from google import genai

load_dotenv(Path.cwd().parent / ".env")
client = genai.Client()
MODEL = "gemini-3.1-flash-lite"

DATA = Path.cwd().parent / "data"
tickets = yaml.safe_load((DATA / "tickets.yml").read_text(encoding="utf-8"))

print(f"key found: {bool(os.getenv('GEMINI_API_KEY'))} · {len(tickets)} tickets · model {MODEL}")

key found: True · 20 tickets · model gemini-3.1-flash-lite


---
## ⭐ Exercise 1 — Token detective

In [2]:
samples = {
    "English": "Where is my order? I ordered headphones last week and the tracking has not moved.",
    "Bangla":  "আমার অর্ডার কোথায়? আমি গত সপ্তাহে হেডফোন অর্ডার করেছি এবং ট্র্যাকিং আপডেট হয়নি।",
    "Python":  "def triage(text: str) -> dict:\n    return {'category': 'refund', 'urgency': 3}",
}

print(f"{'':10} {'tokens':>7} {'chars':>7} {'chars/token':>13}")
results = {}
for name, text in samples.items():
    n = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    results[name] = len(text) / n
    print(f"{name:10} {n:>7} {len(text):>7} {len(text) / n:>13.2f}")

worst = min(results, key=results.get)      # fewest chars per token = most expensive
print(f"\nmost expensive per character: {worst}")

            tokens   chars   chars/token


English         18      81          4.50


Bangla          19      81          4.26


Python          27      78          2.89

most expensive per character: Python


### The answer, and why it's the interesting one

**Python code is the most expensive**, at roughly 2.9 characters per token against about 4.3–4.5 for
prose. Identifiers split apart (`triage` may be two or three tokens), and indentation, colons,
brackets, quotes and arrows each tend to become their own token. Prose is what tokenizers are
optimised for; code is punctuation-dense and fragments.

**English and Bangla come out roughly equal — and most people predict Bangla will be far worse.**

That belief is not stupid, it is *out of date*. On GPT-2-era tokenizers it was emphatically true:
non-Latin scripts were poorly represented in the vocabulary and fragmented to bytes, sometimes costing
5–10× more. Modern Gemini tokenizers are trained on a genuinely multilingual corpus and handle Bangla
efficiently. The blog posts that measured the old behaviour are still the top search results.

So the real lesson is the meta one: **you just contradicted a widely repeated claim in four lines of
code.** Measure your own workload on your own model. Any tokenizer article older than the model you're
using is a historical document.

**Answer to Part C.3:** look at the **pasted order JSON** first. It's only 10% of the inbox by volume
but the most token-dense content in it, so it is over-represented in the bill — and it's also the
easiest to fix, because you can strip the fields the model doesn't need before sending.

### Part D — messages grow linearly, tokens grow quadratically

**Tokens grow far faster, and the shape of the growth is the answer.**

A model has no memory. Every turn, the entire conversation so far is sent through it again. So turn 1
sends 1 message, turn 2 sends 2, turn 20 sends 20 — and the *total* billed across the whole chat is
1 + 2 + 3 + … + 20, which is 210 message-loads for 20 messages.

That's the n(n+1)/2 shape: **messages grow linearly, billed input tokens grow with the square.** A
chat twice as long costs roughly four times as much, not twice.

Nothing about `previous_interaction_id` changes this — it saves you the bookkeeping, not the tokens.
Which is exactly why session 6 spends its time on *managing* history: summarising old turns, dropping
what no longer matters, and deciding deliberately what is worth carrying forward.

In [3]:
# The arithmetic, so you can feel the shape rather than take it on faith.
TOKENS_PER_MESSAGE = 120

for turns in (5, 10, 20, 40):
    billed = sum(range(1, turns + 1)) * TOKENS_PER_MESSAGE
    print(f"{turns:>3} turns → {billed:>7,} input tokens billed "
          f"({billed / (turns * TOKENS_PER_MESSAGE):.1f}× the naive estimate)")

  5 turns →   1,800 input tokens billed (3.0× the naive estimate)
 10 turns →   6,600 input tokens billed (5.5× the naive estimate)
 20 turns →  25,200 input tokens billed (10.5× the naive estimate)
 40 turns →  98,400 input tokens billed (20.5× the naive estimate)


**Comes back in:** S03 (the cost of a schema), S06 (memory management), S08 (chunk size is a token-budget decision).

---
## ⭐⭐ Exercise 2 — Ticket loader

In [4]:
# Part A — count by category
counts: dict[str, int] = {}
for t in tickets:
    counts[t["category"]] = counts.get(t["category"], 0) + 1

for category, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"{category:18} {'█' * n} {n}")
print(f"\ntotal {sum(counts.values())} tickets in {len(counts)} categories")

order_status       ███ 3
refund             ███ 3
complaint          ███ 3
product_question   ███ 3
warranty           ██ 2
billing            ██ 2
shipping           ██ 2
return             █ 1
other              █ 1

total 20 tickets in 9 categories


In [5]:
# Part B — count by customer, same pattern, different key
by_customer: dict[str, int] = {}
for t in tickets:
    by_customer[t["from"]] = by_customer.get(t["from"], 0) + 1

for who, n in sorted(by_customer.items(), key=lambda kv: -kv[1]):
    print(f"{who:24} {'█' * n} {n}")

ayesha@example.com       ██████ 6
tanvir@example.com       ██████ 6
priya@example.com        ████ 4
kamal@example.com        ████ 4


In [6]:
# Part C — one function instead of two loops.
# `field` is a PARAMETER, so the same function answers both questions.
def count_by(tickets: list[dict], field: str) -> dict[str, int]:
    """Count tickets grouped by any field. Unknown field -> KeyError, deliberately loud."""
    counts: dict[str, int] = {}
    for t in tickets:
        counts[t[field]] = counts.get(t[field], 0) + 1
    return counts


print(count_by(tickets, "category"))
print()
print(count_by(tickets, "received"))     # works on any field, for free

{'order_status': 3, 'refund': 3, 'complaint': 3, 'return': 1, 'warranty': 2, 'product_question': 3, 'billing': 2, 'shipping': 2, 'other': 1}

{'2026-08-02': 2, '2026-08-06': 3, '2026-08-04': 3, '2026-08-05': 3, '2026-08-07': 2, '2026-08-03': 2, '2026-08-08': 3, '2026-08-09': 2}


### Why this exercise exists

Three things, in increasing order of importance.

**1. `counts.get(key, 0) + 1` is the update pattern of the whole course.** In session 10 a graph node
receives state, returns a partial update, and the framework merges it. The shape is identical: read
what's there, default if it isn't, write back the combination. If it's second nature now, S10 is a
notation change rather than a new idea.

**2. `count_by(tickets, field)` is a tool, not a script.** Passing the field name as an argument
instead of hard-coding it is the entire difference. Session 5 is about writing functions the *model*
calls, and a tool that only answers one hard-coded question is useless to it.

**3. The deliberate crash.** `t[field]` raises `KeyError` on a bad field name rather than silently
returning nothing. That's the right call here — a typo'd field name is a programming error and should
stop loudly. Contrast with S05's tools, which return `{"error": "not found"}` because a customer
asking about a non-existent order is *legitimate*, not a bug. **Knowing which situation you're in is
the skill**; `[]` versus `.get()` is just how you express it.

**Comes back in:** S05 (tools with parameters), S10 (state updates).

---
## ⭐⭐⭐ Exercise 3 — Multimodal triage

### Part A — the combined call

In [7]:
from typing import Literal

from pydantic import BaseModel, Field

PHOTO = DATA / "images" / "ticket-photo-headphones.jpg"
claim_text = (
    "Subject: left earcup padding torn\n"
    "I've had these headphones about four months. The padding on the left cup has "
    "split open and the foam is coming out. Photo attached. Is this covered?"
)


class PhotoClaim(BaseModel):
    """What ShopWise needs decided before a human ever opens the ticket."""

    product: str = Field(description="What the photo shows, in three words or fewer")
    visible_damage: str = Field(description="One line: what is actually wrong in the image")
    likely_cause: Literal["manufacturing_defect", "normal_wear", "accidental_damage", "cannot_tell"]
    confidence: Literal["low", "medium", "high"]
    route_to: Literal["auto_approve", "human_review", "request_more_photos", "decline"]


photo_b64 = base64.b64encode(PHOTO.read_bytes()).decode("utf-8")

interaction = client.interactions.create(
    model=MODEL,
    input=[
        {"type": "text", "text": f"Assess this ShopWise warranty claim.\n\n{claim_text}"},
        {"type": "image", "data": photo_b64, "mime_type": "image/jpeg"},
    ],
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": PhotoClaim.model_json_schema(),
    },
)

with_photo = PhotoClaim.model_validate_json(interaction.output_text)
print(with_photo)

product='Over-ear headphones' visible_damage='The fabric on the left earcup padding is torn, exposing the interior foam.' likely_cause='normal_wear' confidence='high' route_to='human_review'


Two details that are easy to miss, and both matter later:

- **The two capabilities compose with no extra machinery.** `input=[…]` carries the image, and
  `response_format=…` constrains the output. They are independent arguments to the same call. Almost
  every "AI feature" you will ever ship is a small combination like this one, not a single clever
  trick.
- **`model_validate_json` is doing real work.** If the model invented a `route_to` value outside the
  `Literal`, this line raises instead of handing your refund logic a string it has never seen. That
  is the difference between a demo and something you would put in front of a customer.

### Part B — the same call, evidence removed

In [8]:
interaction_text_only = client.interactions.create(
    model=MODEL,
    input=f"Assess this ShopWise warranty claim.\n\n{claim_text}",
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": PhotoClaim.model_json_schema(),
    },
)

without_photo = PhotoClaim.model_validate_json(interaction_text_only.output_text)

print(f"{'field':16} {'WITH photo':32} {'TEXT ONLY':32}")
print("-" * 82)
for field in PhotoClaim.model_fields:
    a, b = getattr(with_photo, field), getattr(without_photo, field)
    flag = "  " if a == b else "≠ "
    print(f"{flag}{field:14} {str(a)[:30]:32} {str(b)[:30]:32}")

field            WITH photo                       TEXT ONLY                       
----------------------------------------------------------------------------------
  product        Over-ear headphones              Over-ear headphones             
≠ visible_damage The fabric on the left earcup    Left earcup padding has split   
  likely_cause   normal_wear                      normal_wear                     
  confidence     high                             high                            
  route_to       human_review                     human_review                    


### Part C — what the model could not possibly know

**1. What a photograph cannot contain.** At minimum:

- **How old the product is.** The customer says four months; the image says nothing. Wear consistent
  with four months and wear consistent with three years look identical in one frame.
- **How it was used or stored.** Sat on, left in a hot car, cleaned with alcohol, worn eight hours a
  day — all invisible.
- **Whether this is the item ShopWise sold.** No order, no serial number, no proof it isn't a
  different pair, or a photo from the internet.
- **What ShopWise's warranty actually covers.** The policy is a document the model has never read.
  (That is session 8.)

**2. What `confidence` is worth.** Very often it barely moves when you remove the photograph — and
that tells you the field is measuring *fluency*, not *evidence*. The model is not introspecting on
how much information it had; it is producing the token most likely to follow, and confident-sounding
claims are more common in its training data than hedged ones.

**This is the single most important thing in this exercise.** A `confidence` field that the model
generates about its own answer is not a measurement. If you need real confidence, it has to come
from outside the model: did retrieval actually find a matching policy? Does the order ID exist in the
database? Those are checkable. Self-reported confidence is not.

**3. Would you auto-approve on this?** No — and naming *why* is the syllabus:

| Missing | Session that adds it |
|---|---|
| It can't check the order, the purchase date or the serial number | **S05** — tools that reach real data |
| It hasn't read ShopWise's warranty policy and can't quote a clause | **S08–S09** — retrieval and citations |
| Nothing stops it approving money by itself | **S07, S11** — guardrails and human approval |
| No record of why it decided what it decided | **S04** — tracing |

A reasonable v1 removes `auto_approve` from the `Literal` entirely. The model's job is to *prepare*
the decision — product identified, damage described, evidence flagged as sufficient or not — and a
human makes it. That is a genuinely valuable product, and it ships this week rather than never.

### Part D — survey the field

There is no single right answer here; these are the observations worth having made.

**1. arena.ai — why boards disagree.** Models are trained and tuned for different mixes. A model
post-trained hard on code and verified reasoning will climb the coding board and may sit mid-table
overall, where a large share of votes are conversational, creative or multilingual — and where a
warm, well-formatted answer wins. The overall board largely measures *how pleasant a model is to talk
to*. If your job is triaging tickets, the board that matters is the one closest to that job, and if
none of them are close, the only real benchmark is your own tickets. (You build exactly that in
session 4.)

**2. Hugging Face — the licence is the point.** Model cards are wildly variable in quality; the good
ones state training data, intended use and known failure modes. Two things people miss on their first
visit:

- **"Open weights" ≠ "do what you like".** Several major open models carry commercial restrictions,
  acceptable-use clauses, or attribution requirements. Read before you build a business on one.
- **Limitations sections are the most useful part.** A card that admits weak performance on your
  language has just saved you a week.

**3. openrouter.ai — the ratio is the lesson.** Cheap and frontier models typically differ by one to
two orders of magnitude per token. At roughly 100 tokens per triaged ticket and 200 tickets a day
that is about 600k tokens a month — which on a cheap model is small change and on a frontier model is
still not very much.

And that is the real finding: **at this volume, model price is not the interesting variable.** The
right answer to "which model" is "the cheapest one that passes my evaluation set", and the honest
follow-up is that at 200 tickets/day you should spend your effort on correctness and escalation
rather than on saving a few dollars. Cost discipline matters enormously at 200,000 tickets/day and
almost not at all at 200 — and knowing which regime you are in is the skill.

---
## 💼 Interview question sketches

### 1. "Why don't we just use the best model?"

> Because "best" isn't a property of a model, it's a property of a model *on a task*. The overall
> leaderboards mostly measure how much people enjoy chatting with something, and our job is to
> classify a support ticket — those aren't the same skill, and the ranking changes when you switch to
> a category board.
>
> So I'd do it the boring way. Take a hundred of our real tickets, label them by hand, and run three
> or four candidates against them — including a cheap one, because for a narrow task the cheap model
> is often indistinguishable. Then pick on measured accuracy, latency and cost together. Right now
> we're at two hundred tickets a day, so cost barely matters; if we were at two hundred thousand it
> would decide the whole thing.
>
> And I'd want the model to be swappable, so this isn't a one-way door. If something better ships in
> three months I want that to be a config change, not a rewrite.

**What it tests:** whether you evaluate or whether you follow hype. The strongest answers mention an
evaluation set built from *their* data, and mention swappability. "It depends" without naming the
dependencies is a fail.

### 2. Compute the cost of one call

Do this out loud, showing the units — interviewers are checking arithmetic discipline more than the
number.

**One call:**

- input: 2,000 tokens × $0.10 / 1,000,000 = **$0.0002**
- output: 500 tokens × $0.40 / 1,000,000 = **$0.0002**
- **total ≈ $0.0004 per call** (four hundredths of a cent)

**5,000 calls/day for 30 days** = 150,000 calls × $0.0004 = **$60/month**.

In [9]:
# The same arithmetic, so you can change the numbers and see what moves.
IN_RATE, OUT_RATE = 0.10, 0.40      # dollars per 1M tokens (illustrative sheet)
in_tok, out_tok = 2_000, 500

cost_in  = in_tok  * IN_RATE  / 1_000_000
cost_out = out_tok * OUT_RATE / 1_000_000
per_call = cost_in + cost_out

print(f"input   {cost_in:.6f}  ({cost_in/per_call:.0%} of the call)")
print(f"output  {cost_out:.6f}  ({cost_out/per_call:.0%} of the call)")
print(f"per call {per_call:.6f}")
print(f"\n5,000/day × 30 days = {150_000 * per_call:,.2f} per month")

# Halving: attack whichever side dominates. Here they are equal in cost, but the
# input is 4x the tokens — so it is where the compressible waste lives.
print(f"\nif the prompt were 500 tokens instead of 2,000:"
      f" {150_000 * (500*IN_RATE/1e6 + cost_out):,.2f}/month")

input   0.000200  (50% of the call)
output  0.000200  (50% of the call)
per call 0.000400

5,000/day × 30 days = 60.00 per month

if the prompt were 500 tokens instead of 2,000: 37.50/month


**"Which number do you attack?"** — this is the part that separates answers.

Output tokens are 4× more expensive *per token*, so the instinct is to shorten the answer. But look at
the split: the two sides cost the same here, because the prompt is 4× longer. And the prompt is where
the waste usually is — a bloated system prompt, or the entire policy document pasted in when one
paragraph would do.

So: **attack the input**, because it's the compressible one. Shortening the answer degrades the product
your customer sees; shortening the prompt is invisible to them. Concretely — retrieve the relevant
paragraph instead of sending everything (S08), strip fields the model doesn't need, and cache the
static part of the prompt.

The strongest possible answer adds: *"and before I optimise either, I'd check whether a cheaper model
handles this workload — that's usually a bigger win than token trimming."* At $60/month, the honest
answer is also that this bill is not worth an engineer's afternoon; say so, because knowing when *not*
to optimise is seniority.

### 3. "Will our patient data be safe?"

> There are two shapes this can take, and they're genuinely different.
>
> One is a hosted model — OpenAI, Anthropic, Google. You send them the text and they send back an
> answer. It's the most capable option and there's nothing to operate, but the data does leave your
> building, and you're relying on a contract rather than on architecture. On the paid tiers these
> providers commit not to train on your data; on the free tiers, several explicitly do, and that
> difference is the first thing I'd check.
>
> The other is an open-weights model — Llama, Qwen, Mistral — running on hardware you control. The
> data never leaves. You give up some capability and you take on real work: GPUs, uptime, upgrades,
> someone who owns it.
>
> For a hospital I'd start from the assumption that we run it ourselves, because "the data never
> left" is an argument you can make to a regulator and "they promised not to look" is not.
>
> Before I committed, I'd need to know: what regulation actually applies to them and whether it names
> data residency; whether the records can be de-identified before they reach any model at all; and
> what accuracy the task genuinely needs — because if a mid-sized open model is good enough, the
> question answers itself.

**What it tests:** whether you can translate an architectural choice into business risk, and whether
you ask questions before recommending. A candidate who answers only "use open source, it's private"
has skipped the part where you find out what the client is required to do.

**A good follow-up to expect:** "what if the open model isn't accurate enough?" The answer is that you
narrow the task until it is — summarise and draft rather than decide — and keep the human in the loop
for anything consequential. Which, again, is sessions 7 and 11.

---
## 💼 Case study notes — how support AI actually gets rolled out

For discussion. Nothing here is code; all of it is what the meeting sounds like.

**Two numbers in tension.** *Deflection rate* — the share of tickets resolved without a human — is what
the business buys. *CSAT* is what it risks. A rollout that pushes deflection by refusing to escalate
buys the first number by burning the second, and the damage shows up a quarter later, which is exactly
long enough for whoever shipped it to have been promoted.

**Every serious deployment keeps a human path.** Not as a fallback for failures, but as a designed
route with its own SLA. The interesting design question is not "can the AI handle this?" but "what
does it cost us when it wrongly believes it can?"

**What one confidently-wrong answer costs.** A made-up return window quoted to a customer becomes,
in order: a refund you didn't owe, a screenshot on social media, and a policy question from legal
about whether you're bound by what your bot said. That third one is the expensive one — and it's why
"the AI said it" has never once worked as a defence.

**Where the ShopWise build makes each safeguard concrete:** S07 guardrails (it cannot be talked into
ignoring its instructions) · S09 citations (it quotes the policy or refuses) · S11 approvals (money
never moves without a human) · S04 tracing (when it goes wrong, you can prove what happened).